# House Price Prediction Model



## Step 1: Load Dataset
We load the dataset from a CSV file and perform initial preprocessing such as dropping unnecessary columns.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
# Load the dataset
file_path = 'final_dataset.csv'
df = pd.read_csv(file_path)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df.head()

,beds,baths,size,zip_code,price
0,3,2.5,2590.0,98144,795000.0
1,4,2.0,2240.0,98106,915000.0
2,4,3.0,2040.0,98107,950000.0
3,4,3.0,3800.0,98199,1950000.0
4,2,2.0,1042.0,98102,950000.0


## Step 2: Feature Engineering
We create new features like price per square foot, bed-bath ratio, size-zip interaction, and apply logarithmic transformation to size and price.

In [3]:
# Feature Engineering
df['price_per_sqft'] = df['price'] / df['size']
df['bed_bath_ratio'] = df['beds'] / df['baths']
df['size_zip_interaction'] = df['size'] * df['zip_code']
df['beds_baths_interaction'] = df['beds'] * df['baths']
df['log_size'] = np.log1p(df['size'])
df['log_price'] = np.log1p(df['price'])

In [4]:
# Define features and target
X = df[['beds', 'baths', 'log_size', 'zip_code', 'price_per_sqft', 'bed_bath_ratio', 'size_zip_interaction', 'beds_baths_interaction']]
y = df['log_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 3: Data Splitting
The dataset is split into training and testing sets with an 80-20 split.

In [5]:
# Preprocessing pipeline
column_trans = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['zip_code']),
    ('scaler', StandardScaler(), ['beds', 'baths', 'log_size', 'price_per_sqft', 'bed_bath_ratio', 'size_zip_interaction', 'beds_baths_interaction'])
], remainder='passthrough')

# Polynomial Features
poly = PolynomialFeatures(degree=2, include_bias=False)

## Step 4: Data Preprocessing
A preprocessing pipeline is created to scale numerical features and one-hot encode categorical features.

In [6]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=7, objective='reg:squarederror', random_state=42)
}

## Step 5: Model Training and Evaluation
We train multiple regression models including Linear Regression, Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting, and XGBoost.

In [7]:
# Train models and evaluate
results = {}
for name, model in models.items():
    pipeline = make_pipeline(column_trans, poly, model)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R² Score': r2, 'RMSE': rmse}
results

{'Linear Regression': {'R² Score': 0.9520783109467227,
  'RMSE': 0.11228336862712995},
 'Ridge Regression': {'R² Score': 0.9511334869306096,
  'RMSE': 0.11338485534097455},
 'Lasso Regression': {'R² Score': 0.5996818050460329,
  'RMSE': 0.3245279003405691},
 'ElasticNet': {'R² Score': 0.691279721140654, 'RMSE': 0.2849915307180094},
 'Random Forest': {'R² Score': 0.9871327759184563,
  'RMSE': 0.058182423501180754},
 'Gradient Boosting': {'R² Score': 0.9927093732586136,
  'RMSE': 0.04379573377914942},
 'XGBoost': {'R² Score': 0.9917096790917218, 'RMSE': 0.046701953383389054}}

## Step 6: Feature Importance Analysis
We analyze the importance of features using the Gradient Boosting model.

In [8]:
# Feature Importance Analysis using Gradient Boosting
best_model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42)
best_model.fit(X_train, y_train)
feature_importances = best_model.feature_importances_
feature_names = X_train.columns
sorted_features = sorted(zip(feature_importances, feature_names), reverse=True)
for importance, name in sorted_features:
    print(f'{name}: {importance:.4f}')

size_zip_interaction: 0.4275
price_per_sqft: 0.3924
log_size: 0.1785
beds_baths_interaction: 0.0005
bed_bath_ratio: 0.0004
baths: 0.0003
zip_code: 0.0003
beds: 0.0001
